In [ ]:
# === BƯỚC 0: Thiết lập & Tải dữ liệu ===
import os, tarfile
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# 1) Nếu đã có file .tar.gz trong data/, giải nén
TAR_PATH = "data/hwu.tar.gz"
if os.path.exists(TAR_PATH):
    with tarfile.open(TAR_PATH, "r:gz") as tar:
        tar.extractall("data")
    print(" Đã giải nén:", TAR_PATH)
else:
    # 2) Nếu chưa có, cho phép upload trực tiếp 1 file .tar.gz HOẶC 3 file csv (train/val/test)
    try:
        from google.colab import files
        print(" Upload hwu.tar.gz HOẶC train/val/test (.csv)")
        up = files.upload()
        os.makedirs("data", exist_ok=True)
        for name, content in up.items():
            open(os.path.join("data", name), "wb").write(content)
        # Tự giải nén nếu người dùng upload .tar.gz
        for name in up.keys():
            if name.endswith(".tar.gz") or name.endswith(".tgz"):
                with tarfile.open(os.path.join("data", name), "r:gz") as tar:
                    tar.extractall("data")
                print(" Đã giải nén:", name)
    except Exception as e:
        print(" Không chạy trong Colab hoặc upload thất bại:", e)

# 3) Xác định thư mục chứa file sau giải nén
data_dir = "data/hwu" if os.path.isdir("data/hwu") else "data"
print(" data_dir =", data_dir, "| Files:", os.listdir(data_dir))

# 4) Đọc đúng file CSV (theo gói bạn vừa giải nén)
train_path = os.path.join(data_dir, "train.csv")
val_path   = os.path.join(data_dir, "val.csv")
test_path  = os.path.join(data_dir, "test.csv")
assert os.path.exists(train_path) and os.path.exists(val_path) and os.path.exists(test_path), \
       "Không tìm thấy train.csv/val.csv/test.csv trong " + data_dir

# 5) Đọc dữ liệu (CSV, header chuẩn: text + intent). Nếu header khác, đổi tên cột tương ứng.
df_train = pd.read_csv(train_path)
df_val   = pd.read_csv(val_path)
df_test  = pd.read_csv(test_path)

# Chuẩn hoá tên cột nếu khác (ví dụ 'sentence','query' → 'text'; 'label','category' → 'intent')
def normalize_cols(df):
    cols = {c.lower(): c for c in df.columns}
    text_col = None
    for k in ["text","utterance","sentence","query","content"]:
        if k in cols: text_col = cols[k]; break
    intent_col = None
    for k in ["intent","label","category","class","target"]:
        if k in cols: intent_col = cols[k]; break
    if text_col is None: text_col = df.columns[0]
    if intent_col is None: intent_col = df.columns[1]
    return df.rename(columns={text_col:"text", intent_col:"intent"})[["text","intent"]]

df_train = normalize_cols(df_train)
df_val   = normalize_cols(df_val)
df_test  = normalize_cols(df_test)

print("Train shape:", df_train.shape)
print("Validation shape:", df_val.shape)
print("Test shape:", df_test.shape)
display(df_train.head())

# 6) Mã hoá nhãn bằng LabelEncoder (fit TRÊN CẢ train+val+test để đồng bộ bộ nhãn)
le = LabelEncoder()
le.fit(pd.concat([df_train["intent"], df_val["intent"], df_test["intent"]], axis=0))

y_train = le.transform(df_train["intent"])
y_val   = le.transform(df_val["intent"])
y_test  = le.transform(df_test["intent"])
num_classes = len(le.classes_)
print(" num_classes:", num_classes, "| ví dụ nhãn:", list(le.classes_)[:10], "…")


 Upload hwu.tar.gz HOẶC train/val/test (.csv)


Saving hwu.tar.gz to hwu.tar (6).gz
 data_dir = data/hwu | Files: ['train_10.csv', 'categories.json', 'val.csv', 'train_5.csv', 'test.csv', 'train.csv']
Train shape: (8954, 2)
Validation shape: (1076, 2)
Test shape: (1076, 2)


,text,intent
0,what alarms do i have set right now,alarm_query
1,checkout today alarm of meeting,alarm_query
2,report alarm settings,alarm_query
3,see see for me the alarms that you have set to...,alarm_query
4,is there an alarm for ten am,alarm_query


 num_classes: 64 | ví dụ nhãn: ['alarm_query', 'alarm_remove', 'alarm_set', 'audio_volume_down', 'audio_volume_mute', 'audio_volume_up', 'calendar_query', 'calendar_remove', 'calendar_set', 'cooking_recipe'] …


# Nhiệm vụ 1

In [20]:
# === NHIỆM VỤ 1: TF-IDF + Logistic Regression ===
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, f1_score

tfidf_lr_pipeline = make_pipeline(
    TfidfVectorizer(max_features=5000),
    LogisticRegression(max_iter=1000, n_jobs=-1, random_state=42)
)

tfidf_lr_pipeline.fit(df_train["text"], y_train)
y_pred_lr = tfidf_lr_pipeline.predict(df_test["text"])

print("=== Classification report: TF-IDF + LR ===")
print(classification_report(y_test, y_pred_lr, target_names=le.classes_, digits=4))
f1_lr = f1_score(y_test, y_pred_lr, average="macro")
print("Macro-F1 (test):", f1_lr)


=== Classification report: TF-IDF + LR ===
                          precision    recall  f1-score   support

             alarm_query     0.9000    0.9474    0.9231        19
            alarm_remove     1.0000    0.7273    0.8421        11
               alarm_set     0.7727    0.8947    0.8293        19
       audio_volume_down     1.0000    0.7500    0.8571         8
       audio_volume_mute     0.9231    0.8000    0.8571        15
         audio_volume_up     0.9286    1.0000    0.9630        13
          calendar_query     0.4545    0.5263    0.4878        19
         calendar_remove     0.8947    0.8947    0.8947        19
            calendar_set     0.8667    0.6842    0.7647        19
          cooking_recipe     0.5909    0.6842    0.6341        19
        datetime_convert     0.6667    0.7500    0.7059         8
          datetime_query     0.7391    0.8947    0.8095        19
        email_addcontact     0.7778    0.8750    0.8235         8
             email_query     0.8

# Nhiệm vụ 2


In [21]:
# === NHIỆM VỤ 2: Word2Vec Avg + Dense ===
import numpy as np
from gensim.models import Word2Vec
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# 1) Huấn luyện Word2Vec trên văn bản train
sentences = [str(s).split() for s in df_train["text"]]
w2v_dim = 100
w2v = Word2Vec(sentences=sentences, vector_size=w2v_dim, window=5, min_count=1, workers=4, seed=42)

# 2) Hàm chuyển câu -> vector trung bình
def sentence_to_avg_vector(text, model, dim=100):
    toks = str(text).split()
    vecs = [model.wv[t] for t in toks if t in model.wv]
    if not vecs:
        return np.zeros(dim, dtype="float32")
    return np.mean(vecs, axis=0).astype("float32")

def to_matrix(texts, model, dim=100):
    return np.vstack([sentence_to_avg_vector(t, model, dim) for t in texts])

X_train_avg = to_matrix(df_train["text"], w2v, w2v_dim)
X_val_avg   = to_matrix(df_val["text"],   w2v, w2v_dim)
X_test_avg  = to_matrix(df_test["text"],  w2v, w2v_dim)

# 3) Mô hình Dense
model_avg = Sequential([
    Dense(128, activation='relu', input_shape=(w2v_dim,)),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])
model_avg.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0)
_ = model_avg.fit(X_train_avg, y_train, validation_data=(X_val_avg, y_val),
                  epochs=50, batch_size=64, callbacks=[es], verbose=0)

y_pred_avg = np.argmax(model_avg.predict(X_test_avg, verbose=0), axis=1)
print("=== Classification report: W2V-Avg + Dense ===")
print(classification_report(y_test, y_pred_avg, target_names=le.classes_, digits=4))
from sklearn.metrics import f1_score
f1_avg = f1_score(y_test, y_pred_avg, average="macro")
print("Macro-F1 (test):", f1_avg)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


=== Classification report: W2V-Avg + Dense ===
                          precision    recall  f1-score   support

             alarm_query     0.1887    0.5263    0.2778        19
            alarm_remove     1.0000    0.0909    0.1667        11
               alarm_set     0.3864    0.8947    0.5397        19
       audio_volume_down     0.6667    0.2500    0.3636         8
       audio_volume_mute     0.0000    0.0000    0.0000        15
         audio_volume_up     0.1538    0.1538    0.1538        13
          calendar_query     0.0000    0.0000    0.0000        19
         calendar_remove     0.3333    0.3158    0.3243        19
            calendar_set     0.0000    0.0000    0.0000        19
          cooking_recipe     0.2500    0.0526    0.0870        19
        datetime_convert     0.0000    0.0000    0.0000         8
          datetime_query     0.1368    0.6842    0.2281        19
        email_addcontact     0.0000    0.0000    0.0000         8
             email_query    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Nhiệm vụ 3

In [22]:
# === NHIỆM VỤ 3: Embedding (pre-trained) + LSTM ===
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM
from tensorflow.keras.models import Sequential

# 1) Tokenizer + Padding
max_words = 20000
oov_tok = "<UNK>"
tokenizer = Tokenizer(num_words=max_words, oov_token=oov_tok)
tokenizer.fit_on_texts(df_train["text"])

def to_pad(texts, tok, max_len=50):
    seqs = tok.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=max_len, padding='post', truncating='post')

max_len = 50
X_train_pad = to_pad(df_train["text"], tokenizer, max_len)
X_val_pad   = to_pad(df_val["text"],   tokenizer, max_len)
X_test_pad  = to_pad(df_test["text"],  tokenizer, max_len)

vocab_size = min(max_words, len(tokenizer.word_index) + 1)
embedding_dim = w2v_dim

# 2) Ma trận embedding từ W2V (đã train ở Nhiệm vụ 2)
embedding_matrix = np.zeros((vocab_size, embedding_dim), dtype="float32")
for word, idx in tokenizer.word_index.items():
    if idx < vocab_size and word in w2v.wv:
        embedding_matrix[idx] = w2v.wv[word]

# 3) LSTM với embedding pre-trained (đóng băng)
lstm_pre = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim,
              weights=[embedding_matrix], input_length=max_len, trainable=False),
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),
    Dense(num_classes, activation='softmax')
])
lstm_pre.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

es = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0)
_ = lstm_pre.fit(X_train_pad, y_train, validation_data=(X_val_pad, y_val),
                 epochs=30, batch_size=64, callbacks=[es], verbose=0)

y_pred_pre = np.argmax(lstm_pre.predict(X_test_pad, verbose=0), axis=1)
print("=== Classification report: Emb(pretrained) + LSTM ===")
print(classification_report(y_test, y_pred_pre, target_names=le.classes_, digits=4))
f1_pre = f1_score(y_test, y_pred_pre, average="macro")
print("Macro-F1 (test):", f1_pre)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


=== Classification report: Emb(pretrained) + LSTM ===
                          precision    recall  f1-score   support

             alarm_query     0.0000    0.0000    0.0000        19
            alarm_remove     0.0000    0.0000    0.0000        11
               alarm_set     0.0884    0.6842    0.1566        19
       audio_volume_down     0.0000    0.0000    0.0000         8
       audio_volume_mute     0.0000    0.0000    0.0000        15
         audio_volume_up     0.0000    0.0000    0.0000        13
          calendar_query     0.0000    0.0000    0.0000        19
         calendar_remove     0.0000    0.0000    0.0000        19
            calendar_set     0.0000    0.0000    0.0000        19
          cooking_recipe     0.0000    0.0000    0.0000        19
        datetime_convert     0.0000    0.0000    0.0000         8
          datetime_query     0.0098    0.0526    0.0165        19
        email_addcontact     0.0000    0.0000    0.0000         8
             email_qu

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Nhiệm vụ 4

In [23]:
# === NHIỆM VỤ 4: Embedding (scratch) + LSTM ===
lstm_scr = Sequential([
    Embedding(input_dim=vocab_size, output_dim=100, input_length=max_len),
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),
    Dense(num_classes, activation='softmax')
])
lstm_scr.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

es = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0)
_ = lstm_scr.fit(X_train_pad, y_train, validation_data=(X_val_pad, y_val),
                 epochs=30, batch_size=64, callbacks=[es], verbose=0)

y_pred_scr = np.argmax(lstm_scr.predict(X_test_pad, verbose=0), axis=1)
print("=== Classification report: Emb(scratch) + LSTM ===")
print(classification_report(y_test, y_pred_scr, target_names=le.classes_, digits=4))
f1_scr = f1_score(y_test, y_pred_scr, average="macro")
print("Macro-F1 (test):", f1_scr)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


=== Classification report: Emb(scratch) + LSTM ===
                          precision    recall  f1-score   support

             alarm_query     0.0000    0.0000    0.0000        19
            alarm_remove     0.0000    0.0000    0.0000        11
               alarm_set     0.2895    0.5789    0.3860        19
       audio_volume_down     0.0000    0.0000    0.0000         8
       audio_volume_mute     0.0000    0.0000    0.0000        15
         audio_volume_up     0.0000    0.0000    0.0000        13
          calendar_query     0.0000    0.0000    0.0000        19
         calendar_remove     0.3235    0.5789    0.4151        19
            calendar_set     0.2105    0.2105    0.2105        19
          cooking_recipe     0.1220    0.2632    0.1667        19
        datetime_convert     0.0000    0.0000    0.0000         8
          datetime_query     0.1515    0.5263    0.2353        19
        email_addcontact     0.0000    0.0000    0.0000         8
             email_query

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Nhiệm vụ 5

In [24]:
# === NHIỆM VỤ 5: So sánh định lượng + Phân tích định tính ===
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, classification_report

# ----- 1️ So sánh định lượng (bảng tổng hợp F1 và Loss) -----
loss_avg, _ = model_avg.evaluate(X_test_avg, y_test, verbose=0)
loss_pre, _ = lstm_pre.evaluate(X_test_pad, y_test, verbose=0)
loss_scr, _ = lstm_scr.evaluate(X_test_pad, y_test, verbose=0)

summary = pd.DataFrame({
    "Pipeline": [
        "TF-IDF + Logistic Regression",
        "Word2Vec (Avg) + Dense",
        "Embedding (Pre-trained) + LSTM",
        "Embedding (Scratch) + LSTM"
    ],
    "F1-macro (test)": [f1_lr, f1_avg, f1_pre, f1_scr],
    "Test Loss": [None, loss_avg, loss_pre, loss_scr]
}).sort_values("F1-macro (test)", ascending=False).reset_index(drop=True)

display(summary)

# ----- 2️ Phân tích định tính: các câu “khó” -----
hard_texts = [
    "can you remind me to not call my mom",        # phủ định
    "is it going to be sunny or rainy tomorrow",   # lựa chọn
    "find a flight from new york to london but not through paris"  # phủ định + điều kiện
]

# Dự đoán từ 4 mô hình
def predict_all(texts):
    pred_lr = tfidf_lr_pipeline.predict(texts)
    Xavg = np.vstack([sentence_to_avg_vector(t, w2v, w2v_dim) for t in texts])
    pred_avg = np.argmax(model_avg.predict(Xavg, verbose=0), axis=1)
    seqs = tokenizer.texts_to_sequences(texts)
    Xpad = pad_sequences(seqs, maxlen=max_len, padding='post', truncating='post')
    pred_pre = np.argmax(lstm_pre.predict(Xpad, verbose=0), axis=1)
    pred_scr = np.argmax(lstm_scr.predict(Xpad, verbose=0), axis=1)
    return pred_lr, pred_avg, pred_pre, pred_scr

pred_lr, pred_avg, pred_pre, pred_scr = predict_all(hard_texts)

# Lấy nhãn thật (nếu có trong test set, nếu không sẽ ghi “N/A”)
true_labels = []
for t in hard_texts:
    match = df_test[df_test["text"].str.lower() == t.lower()]
    if len(match) > 0:
        true_labels.append(le.transform(match["intent"])[0])
    else:
        true_labels.append(None)

# Bảng kết quả định tính
rows = []
for i, text in enumerate(hard_texts):
    y_true = true_labels[i]
    row = {
        "text": text,
        "True Intent": le.classes_[y_true] if y_true is not None else "N/A",
        "TF-IDF + LR": le.classes_[pred_lr[i]],
        "W2V-Avg + Dense": le.classes_[pred_avg[i]],
        "LSTM (pre)": le.classes_[pred_pre[i]],
        "LSTM (scratch)": le.classes_[pred_scr[i]],
    }
    # Thêm dấu ✓ nếu có nhãn thật và dự đoán đúng
    if y_true is not None:
        row["✓ LR"] = "✓" if pred_lr[i] == y_true else "✗"
        row["✓ W2V"] = "✓" if pred_avg[i] == y_true else "✗"
        row["✓ LSTM-pre"] = "✓" if pred_pre[i] == y_true else "✗"
        row["✓ LSTM-scr"] = "✓" if pred_scr[i] == y_true else "✗"
    rows.append(row)

qual_df = pd.DataFrame(rows)
display(qual_df)

# ----- 3️ Tổng kết nhận xét -----
print("\n Nhận xét:")
print("- Các câu có phủ định ('not') hoặc mệnh đề phụ ('or', 'but not through ...') thường khiến mô hình TF-IDF và W2V trung bình dự đoán sai, "
      "vì chúng không nắm được thứ tự và phạm vi của phủ định.")
print("- LSTM (đặc biệt bản pre-trained) có xu hướng hiểu ngữ cảnh tốt hơn, vì trạng thái ẩn theo chuỗi giúp mô hình nhận diện được mối quan hệ ngữ pháp và ý phủ định.")
print("- Nếu LSTM (pre-trained) chính xác hơn bản học từ đầu, điều đó chứng tỏ embedding Word2Vec cung cấp ngữ nghĩa ban đầu hữu ích, giúp mô hình hội tụ nhanh và tổng quát tốt hơn.")


,Pipeline,F1-macro (test),Test Loss
0,TF-IDF + Logistic Regression,0.835298,NaN
1,Embedding (Scratch) + LSTM,0.149525,2.941058
2,Word2Vec (Avg) + Dense,0.149018,3.032118
3,Embedding (Pre-trained) + LSTM,0.036167,3.603505


,text,True Intent,TF-IDF + LR,W2V-Avg + Dense,LSTM (pre),LSTM (scratch)
0,can you remind me to not call my mom,N/A,calendar_set,general_joke,general_dontcare,email_sendemail
1,is it going to be sunny or rainy tomorrow,N/A,weather_query,calendar_query,lists_createoradd,social_post
2,find a flight from new york to london but not ...,N/A,general_negate,transport_query,alarm_set,cooking_recipe



 Nhận xét:
- Các câu có phủ định ('not') hoặc mệnh đề phụ ('or', 'but not through ...') thường khiến mô hình TF-IDF và W2V trung bình dự đoán sai, vì chúng không nắm được thứ tự và phạm vi của phủ định.
- LSTM (đặc biệt bản pre-trained) có xu hướng hiểu ngữ cảnh tốt hơn, vì trạng thái ẩn theo chuỗi giúp mô hình nhận diện được mối quan hệ ngữ pháp và ý phủ định.
- Nếu LSTM (pre-trained) chính xác hơn bản học từ đầu, điều đó chứng tỏ embedding Word2Vec cung cấp ngữ nghĩa ban đầu hữu ích, giúp mô hình hội tụ nhanh và tổng quát tốt hơn.
